# 04. Kết quả và biểu đồ

Chỉ đọc file trong `data/processed/` và `artifacts/`, chạy trên CPU trong vài giây. Hình được lưu vào `docs/figures/`.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "https://github.com/vinhlee9x/conflict-detection-obsolescence-forecasting.git", "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not os.path.exists("src") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Thư mục làm việc:", os.getcwd())

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd
from src import plots
from src.io_utils import read_json, read_npz, read_parquet
from src.paths import ARTIFACTS, DATA_PROCESSED, FIGURES
kb = ARTIFACTS / "kb"
pairs = read_parquet(DATA_PROCESSED / "kb_pairs.parquet")
scored = read_parquet(kb / "scored_pairs.parquet")
grid = pd.read_csv(kb / "grid_results.csv")
config = read_json(kb / "config.json")
layer2 = read_json(kb / "layer2_results.json")
layer3 = read_json(kb / "layer3_results.json")
predictions = read_parquet(kb / "forecast_test_predictions.parquet")
shap_data = read_npz(kb / "shap_values.npz")
def show(fig, name):
    plots.save(fig, FIGURES / f"{name}.png")
    display(fig)

## 1. Dataset

In [ ]:
show(plots.dataset_overview(pairs), "01_dataset_overview")
pd.DataFrame(layer2["counts"])

## 2. Động lực: cosine trên ViNLI

In [ ]:
vinli = read_parquet(ARTIFACTS / "vinli" / "vinli_cosine.parquet")
vinli_summary = read_json(ARTIFACTS / "vinli" / "vinli_summary.json")
show(plots.vinli_cosine(vinli, vinli_summary), "02_vinli_cosine")
vinli_summary

## 3. Layer 2: grid search λ×τ và ablation

In [ ]:
show(plots.tenant_heatmaps(grid, config), "03_grid_heatmaps")

In [ ]:
show(plots.lambda_profile(grid), "04_lambda_profile")

In [ ]:
table = pd.DataFrame(layer2["test"])
table[["scope", "method_label", "lambda", "tau", "precision", "recall", "f1", "f1_ci95", "pr_auc"]].round(3)

In [ ]:
show(plots.pr_curves(scored), "05_pr_curves")

In [ ]:
show(plots.signal_scatter(scored), "06_signal_scatter")

In [ ]:
for tenant in sorted(scored.tenant.unique()):
    show(plots.score_distribution(scored, tenant), f"07_score_distribution_{tenant}")
pd.DataFrame(layer2["test_by_variant"]).round(2)

## 4. Layer 3: dự báo lỗi thời

In [ ]:
pd.DataFrame(layer3["test"]).round(3)

In [ ]:
models = [row["model"] for row in layer3["test"]]
show(plots.forecast_curves(predictions, models), "08_forecast_curves")

In [ ]:
show(plots.alpha_selection(layer3), "09_alpha_selection")
show(plots.learning_curve(layer3), "10_learning_curve")

In [ ]:
show(plots.shap_summary(shap_data), "11_shap_summary")
top = read_parquet(kb / "forecast_scores.parquet").head(5)
display(top[["doc_id", "tenant", "topic", "age_days", "n_open_candidates", "n_confirmed_conflicts", "p_obsolete"]])
show(plots.shap_waterfall(shap_data, top.doc_id.iloc[0]), "12_shap_waterfall")

## 5. Tuỳ chọn: tương tác

Chọn một cặp để xem s, p, score; kéo λ, τ để thấy P/R/F1 trên test thay đổi.

In [ ]:
import ipywidgets as widgets
docs = read_parquet(DATA_PROCESSED / "kb_docs.parquet").set_index("doc_id")
test = scored[scored.split == "test"].reset_index(drop=True)
options = [(f"{r.doc_a} ↔ {r.doc_b} ({plots.VARIANT_LABELS[r.variant_type]})", i) for i, r in test.iterrows()]
def inspect_pair(i):
    r = test.loc[i]
    verdict = "CÓ thể mâu thuẫn" if r.pred_fusion else "không vượt ngưỡng"
    print(f"s = {r.s:.3f}  p = {r.p:.3f}  score = {r.score_fusion:.3f}  τ = {r.tau_fusion:.2f}  →  {verdict}  (nhãn thật: {r.label})")
    print(docs.loc[r.doc_a, "text"]); print("-" * 60); print(docs.loc[r.doc_b, "text"])
widgets.interact(inspect_pair, i=widgets.Dropdown(options=options, description="Cặp"))

In [ ]:
def tune(tenant, lam, tau):
    m = plots.threshold_metrics(scored, lam, tau, tenant=None if tenant == "all" else tenant)
    print(f"P = {m['precision']:.3f}  R = {m['recall']:.3f}  F1 = {m['f1']:.3f}   TP={m['tp']} FP={m['fp']} FN={m['fn']} TN={m['tn']}")
widgets.interact(tune, tenant=["all", "tech", "policy"],
                 lam=widgets.FloatSlider(value=config["global"]["FUSION_LAMBDA"], min=0, max=1, step=0.05, description="λ"),
                 tau=widgets.FloatSlider(value=config["global"]["CONFLICT_THRESHOLD_TAU"], min=0.3, max=0.95, step=0.01, description="τ"))